# New Zen Chan Graph Ideas

In [1]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go

## 1. Total Browsing Time by Day/Month/Year (Bar Chart)

In [2]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, visit_duration_sec FROM visits', conn)
conn.close()

df['visit_datetime'] = pd.to_datetime(df['visit_datetime'])

# Daily
daily_browsing = df.groupby(df['visit_datetime'].dt.date)['visit_duration_sec'].sum().reset_index()
fig_daily = px.bar(daily_browsing, x='visit_datetime', y='visit_duration_sec', title='Total Browsing Time by Day')
fig_daily.show()

# Monthly
monthly_browsing = df.groupby(df['visit_datetime'].dt.to_period('M'))['visit_duration_sec'].sum().reset_index()
monthly_browsing['visit_datetime'] = monthly_browsing['visit_datetime'].astype(str)
fig_monthly = px.bar(monthly_browsing, x='visit_datetime', y='visit_duration_sec', title='Total Browsing Time by Month')
fig_monthly.show()

# Yearly
yearly_browsing = df.groupby(df['visit_datetime'].dt.year)['visit_duration_sec'].sum().reset_index()
fig_yearly = px.bar(yearly_browsing, x='visit_datetime', y='visit_duration_sec', title='Total Browsing Time by Year')
fig_yearly.show()

## 2. Top Moods/Activities for a Given Period (Bar Chart)

In [3]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, mood, pre_labels, visit_duration_sec FROM visits', conn)
conn.close()

df['visit_datetime'] = pd.to_datetime(df['visit_datetime'])

def get_top_data(dataframe, column, time_filter=None):
    filtered_df = dataframe.copy()
    if time_filter == 'today':
        filtered_df = filtered_df[filtered_df['visit_datetime'].dt.date == pd.to_datetime('today').date()]
    elif time_filter == 'this_month':
        filtered_df = filtered_df[filtered_df['visit_datetime'].dt.to_period('M') == pd.to_datetime('today').to_period('M')]
    elif time_filter == 'this_year':
        filtered_df = filtered_df[filtered_df['visit_datetime'].dt.year == pd.to_datetime('today').year]
    
    if not filtered_df.empty:
        top_data = filtered_df.groupby(column)['visit_duration_sec'].sum().nlargest(5).reset_index()
        return top_data
    return pd.DataFrame(columns=[column, 'visit_duration_sec'])

time_periods = [None, 'today', 'this_month', 'this_year']
columns_to_analyze = ['mood', 'pre_labels']

for period in time_periods:
    for col in columns_to_analyze:
        top_data = get_top_data(df, col, period)
        if not top_data.empty:
            title_suffix = f' for {period.replace("_", " ").title()}' if period else ' (All Time)'
            fig = px.bar(top_data, x='visit_duration_sec', y=col, orientation='h',
                         title=f'Top 5 {col.replace("_", " ").title()}{title_suffix}',
                         labels={'visit_duration_sec': 'Total Duration (seconds)', col: col.replace("_", " ").title()})
            fig.show()
        else:
            print(f"No data for Top 5 {col.replace('_', ' ').title()}{' for ' + period.replace('_', ' ').title() if period else ' (All Time)'}")

## 3. Average Visit Duration by Domain (Bar Chart)

In [4]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT domain, visit_duration_sec FROM visits', conn)
conn.close()

avg_duration_by_domain = df.groupby('domain')['visit_duration_sec'].mean().reset_index()
avg_duration_by_domain = avg_duration_by_domain.sort_values(by='visit_duration_sec', ascending=False).head(10)

fig = px.bar(avg_duration_by_domain, x='domain', y='visit_duration_sec',
             title='Average Visit Duration by Domain (Top 10)',
             labels={'domain': 'Domain', 'visit_duration_sec': 'Average Visit Duration (seconds)'})
fig.show()

## 4. New Domains Discovered Over Time (Line Chart)

In [5]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT visit_datetime, domain FROM visits', conn)
conn.close()

df['visit_datetime'] = pd.to_datetime(df['visit_datetime'])
df = df.sort_values('visit_datetime')

unique_domains = set()
new_domains_over_time = []

for index, row in df.iterrows():
    if row['domain'] not in unique_domains:
        unique_domains.add(row['domain'])
    new_domains_over_time.append({'date': row['visit_datetime'], 'new_unique_domains': len(unique_domains)})

new_domains_df = pd.DataFrame(new_domains_over_time)
new_domains_df = new_domains_df.groupby(new_domains_df['date'].dt.date)['new_unique_domains'].max().reset_index()

fig = px.line(new_domains_df, x='date', y='new_unique_domains',
             title='Cumulative New Domains Discovered Over Time',
             labels={'date': 'Date', 'new_unique_domains': 'Cumulative New Unique Domains'})
fig.show()

## 5. Mood/Activity Distribution by Time of Day (Stacked Bar Chart)

In [6]:
conn = sqlite3.connect('data.db')
df = pd.read_sql_query('SELECT time_of_day, mood, pre_labels, visit_duration_sec FROM visits', conn)
conn.close()

order_of_day = ['deep night', 'early morning', 'morning', 'afternoon', 'evening', 'late night']

# Mood Distribution
mood_distribution = df.groupby(['time_of_day', 'mood'])['visit_duration_sec'].sum().reset_index()
mood_distribution['time_of_day'] = pd.Categorical(mood_distribution['time_of_day'], categories=order_of_day, ordered=True)
mood_distribution = mood_distribution.sort_values('time_of_day')

fig_mood = px.bar(mood_distribution, x='time_of_day', y='visit_duration_sec', color='mood',
                  title='Mood Distribution by Time of Day',
                  labels={'time_of_day': 'Time of Day', 'visit_duration_sec': 'Total Duration (seconds)'})
fig_mood.show()

# Activity Distribution
activity_distribution = df.groupby(['time_of_day', 'pre_labels'])['visit_duration_sec'].sum().reset_index()
activity_distribution['time_of_day'] = pd.Categorical(activity_distribution['time_of_day'], categories=order_of_day, ordered=True)
activity_distribution = activity_distribution.sort_values('time_of_day')

fig_activity = px.bar(activity_distribution, x='time_of_day', y='visit_duration_sec', color='pre_labels',
                      title='Activity Distribution by Time of Day',
                      labels={'time_of_day': 'Time of Day', 'visit_duration_sec': 'Total Duration (seconds)'})
fig_activity.show()